In [1]:
import torch
import torch.nn as nn

import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import MNIST

In [2]:
transform = torchvision.transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train = MNIST(root="./data", train=True, download=True, transform=transform)
test = MNIST(root="./data", train=False, download=True, transform=transform)

In [3]:
from torch.utils.data import DataLoader
import torch.optim as optim

train_loader = DataLoader(train, batch_size=64, shuffle=True)
test_loader = DataLoader(test, batch_size=64, shuffle=True)

In [4]:
class CNN(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        self.fc_layer = nn.Sequential(
            nn.Linear(3*3*128, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc_layer(x)

In [5]:
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [6]:
epochs = 15

for epoch in range(epochs):
    model.train()
    training_loss = 0.0

    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        training_loss += loss.item()

    print(f"Epoch = {epoch + 1} / {epochs} | Loss = {training_loss / len(train_loader)}")

Epoch = 1 / 15 | Loss = 0.15595833104682652
Epoch = 2 / 15 | Loss = 0.04166201027183444
Epoch = 3 / 15 | Loss = 0.028856436951087912
Epoch = 4 / 15 | Loss = 0.02261764429685752
Epoch = 5 / 15 | Loss = 0.019136912771058504
Epoch = 6 / 15 | Loss = 0.01525903991443017
Epoch = 7 / 15 | Loss = 0.012576715167825143
Epoch = 8 / 15 | Loss = 0.011003524739741083
Epoch = 9 / 15 | Loss = 0.008812376771853696
Epoch = 10 / 15 | Loss = 0.008401340147546276
Epoch = 11 / 15 | Loss = 0.008549696551733559
Epoch = 12 / 15 | Loss = 0.005997450388869579
Epoch = 13 / 15 | Loss = 0.00846054058606071
Epoch = 14 / 15 | Loss = 0.00656045662732485
Epoch = 15 / 15 | Loss = 0.005476086515327068


In [7]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size = 64, num_layer=2):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layer = num_layer

        self.rnn = nn.RNN(input_size, hidden_size, num_layers=num_layer, batch_first=True)

        self.fc_layer = nn.Sequential(
            nn.Linear(hidden_size, 10)
        )

    def forward(self, x):
        x = x.squeeze()
        h0 = torch.zeros(self.num_layer, x.size(0), self.hidden_size)
        out, _ = self.rnn(x, h0)
        return self.fc_layer(out[:, -1, :])

In [8]:
model_rnn = RNN(28)
optimizer = optim.Adam(model_rnn.parameters())

In [9]:
for epoch in range(epochs):
    model_rnn.train()
    running_loss = 0.0

    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch = {epoch + 1}/{epochs} | Loss = {running_loss / len(train_loader)}")

Epoch = 1/15 | Loss = 0.002324854087303471
Epoch = 2/15 | Loss = 0.002324743683635886
Epoch = 3/15 | Loss = 0.0023247449939044116
Epoch = 4/15 | Loss = 0.0023252381009356525
Epoch = 5/15 | Loss = 0.0023279696380790003
Epoch = 6/15 | Loss = 0.002326642502179302
Epoch = 7/15 | Loss = 0.00233938746774304
Epoch = 8/15 | Loss = 0.002324751408665406
Epoch = 9/15 | Loss = 0.0023247547968879645
Epoch = 10/15 | Loss = 0.002324782233378404
Epoch = 11/15 | Loss = 0.002324750974812511
Epoch = 12/15 | Loss = 0.0023247429263960127
Epoch = 13/15 | Loss = 0.002324742907819437
Epoch = 14/15 | Loss = 0.0023247436045319395
Epoch = 15/15 | Loss = 0.002324743037940837


In [10]:
model.eval()
model_rnn.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)

        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    
    print("Accuracy CNN:\t", correct/total * 100)

    for images, labels in test_loader:
        outputs = model_rnn(images)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    print("Accuracy RNN:\t", correct/total * 100)

Accuracy CNN:	 99.27
Accuracy RNN:	 54.535
